# Final Gold audit (`09_gold_final_audit`)

**Question.** Do the six Gold tables agree with Silver, and with each other?

This notebook runs after the six Gold tables are built. It reads them, compares them with Silver and with each other, writes **one row** into the Gold audit table, and stops with an error when a check failed. It changes no Gold table.

Contract: `docs/gold_contracts.md`, section 15. The plan calls it step 11.

| | |
|---|---|
| Reads | The nine Silver tables, the six Gold tables, and the Gold audit table (for the last successful run of G5) |
| Writes | One row in `audit.gold_pipeline_runs`. Nothing else |

## The six Gold tables

| | Table | One row per | Built from |
|---|---|---|---|
| G1 | `repository_delivery_metrics` | Repository and day | Pull request events |
| G2 | `repository_issue_flow` | Repository and day | Issue events |
| G3 | `repository_contributor_concentration` | Repository and day | Six kinds: push, pull request, issue, issue comment, review, inline review comment |
| G4 | `repository_collaboration_metrics` | Repository and day | Three kinds: issue comment, review, inline review comment |
| G5 | `release_activity_shift` | Release | Release events and the activity around them |
| G6 | `repository_activity_profile` | Repository and day | All nine kinds |

The tables overlap. G6 counts every kind of event, so its counts must agree with the other daily tables on every repository and day. That is what this notebook proves.

## The 33 checks

Every check has a name. When a check fails, its name goes into `failed_checks` of the audit row, like the quality gates of the Gold notebooks.

| Group | Checks | What must be true |
|---|---|---|
| Keys | 12 | Every Gold table: no key twice (`..._KEY_NOT_UNIQUE`), no key with NULL (`..._KEY_IS_NULL`) |
| Gold against Silver | 12 | G1, G2, G3, G4 and G6: the rows are the repository-days of their Silver tables, and the events add up to the Silver rows. G6: each of the nine kinds. G5: its rows plus the release events it left out are all release events of Silver |
| Gold against Gold | 9 | On every repository and day, G6 has the same counts as G1, G2, G3 and G4, and at least as many people as G3. G3 has at least as many people as G4. Every row of G1 to G4 and every repository of G5 is in G6 |

## The steps

| # | Step | What it does |
|---|---|---|
| 1 | Parameters | Reads the widgets and chooses the tables |
| 2 | The days to compare | From `source_date`: one source date or all dates |
| 3 | Read Silver | Counts the Silver rows and the repository-days of those days |
| 4 | Keys | 12 checks |
| 5 | Gold against Silver | 12 checks |
| 6 | Gold against Gold | 9 checks |
| 7 | The result | All checks in one table, and the status of the run |
| 8 | Audit | Writes one audit row. Stops with an error if a check failed |
| 9 | Results | Shows the audit row and the numbers of each table |

Run it after the six Gold notebooks. In the Job it is the last task.

## 1. Parameters

The notebook has the three widgets every Gold notebook has.

| Widget | Blank (default) | With a value |
|---|---|---|
| `source_date` | The whole tables are compared | `YYYY-MM-DD`: only the days of that source date are compared |
| `test_schema` | Production tables | `test_<name>`: every table is read from `github_lakehouse.test_<name>`, and the audit row is written there |
| `run_id` | A generated `manual-...` value | The Job passes its run ID |

### 1.1 Imports and fixed values

In [0]:
import re
import uuid
from datetime import datetime, timezone

# When this run started. It goes into the audit row.
run_started_at = datetime.now(timezone.utc)

# Every time in this notebook is UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

PIPELINE_NAME = "gold_final_audit"

# The pipeline name G5 writes into its audit rows. Step 5.3 reads its last successful run.
G5_PIPELINE_NAME = "gold_release_activity_shift"

### 1.2 `source_date`

Blank means all dates. A value must be a real date in the form `2024-02-03`. Anything else stops the notebook here, before it reads or writes a table.

In [0]:
dbutils.widgets.text("source_date", "")
source_date = dbutils.widgets.get("source_date").strip()

if source_date == "":
    source_date_value = None
else:
    # strptime fails for text that is not a real date, for example 2024-02-31.
    try:
        source_date_value = datetime.strptime(source_date, "%Y-%m-%d").date()
    except ValueError:
        raise ValueError(
            f"Invalid source_date '{source_date}'. "
            "Use a real date such as 2024-02-03, or leave it blank."
        )

    # strptime also accepts 2024-2-3. The text must look exactly like 2024-02-03.
    if str(source_date_value) != source_date:
        raise ValueError(
            f"Invalid source_date '{source_date}'. "
            "Use the form YYYY-MM-DD, for example 2024-02-03."
        )

if source_date_value is None:
    processing_scope = "ALL_DATES"
else:
    processing_scope = "SINGLE_DATE"

print("source_date:", source_date_value)
print("processing_scope:", processing_scope)

### 1.3 `test_schema` and the table names

The table names are set **only** in this cell: the nine Silver tables, the six Gold tables and the audit table. Later cells never change them, otherwise a test run could read production.

In a test schema, the Silver tables, the Gold tables and the audit table are all in that one schema.

A test schema name must start with `test_` and may contain only lower-case letters, digits and underscores.

In [0]:
dbutils.widgets.text("test_schema", "")
test_schema = dbutils.widgets.get("test_schema").strip()

if test_schema == "":
    SILVER_SCHEMA = "github_lakehouse.silver"
    GOLD_SCHEMA = "github_lakehouse.gold"
    AUDIT_TABLE = "github_lakehouse.audit.gold_pipeline_runs"
else:
    if re.fullmatch("test_[a-z0-9_]+", test_schema) is None:
        raise ValueError(
            f"Invalid test_schema '{test_schema}'. "
            "It must start with test_ and may contain only lower-case letters, digits and underscores."
        )
    SILVER_SCHEMA = f"github_lakehouse.{test_schema}"
    GOLD_SCHEMA = f"github_lakehouse.{test_schema}"
    AUDIT_TABLE = f"github_lakehouse.{test_schema}.gold_pipeline_runs"

# The nine Silver tables.
PUSH_TABLE = SILVER_SCHEMA + ".push_events"
PULL_REQUEST_TABLE = SILVER_SCHEMA + ".pull_request_events"
ISSUE_TABLE = SILVER_SCHEMA + ".issue_events"
ISSUE_COMMENT_TABLE = SILVER_SCHEMA + ".issue_comment_events"
REVIEW_TABLE = SILVER_SCHEMA + ".pull_request_review_events"
REVIEW_COMMENT_TABLE = SILVER_SCHEMA + ".pull_request_review_comment_events"
WATCH_TABLE = SILVER_SCHEMA + ".watch_events"
FORK_TABLE = SILVER_SCHEMA + ".fork_events"
RELEASE_TABLE = SILVER_SCHEMA + ".release_events"

# The six Gold tables.
G1_TABLE = GOLD_SCHEMA + ".repository_delivery_metrics"
G2_TABLE = GOLD_SCHEMA + ".repository_issue_flow"
G3_TABLE = GOLD_SCHEMA + ".repository_contributor_concentration"
G4_TABLE = GOLD_SCHEMA + ".repository_collaboration_metrics"
G5_TABLE = GOLD_SCHEMA + ".release_activity_shift"
G6_TABLE = GOLD_SCHEMA + ".repository_activity_profile"

print("Silver schema:", SILVER_SCHEMA)
print("G1           :", G1_TABLE)
print("G2           :", G2_TABLE)
print("G3           :", G3_TABLE)
print("G4           :", G4_TABLE)
print("G5           :", G5_TABLE)
print("G6           :", G6_TABLE)
print("Audit table  :", AUDIT_TABLE)

### 1.4 `run_id`

The Job passes its run ID. A run by hand gets a generated ID. It may contain only letters, digits, `-` and `_`.

In [0]:
dbutils.widgets.text("run_id", "")
run_id = dbutils.widgets.get("run_id").strip()

if run_id == "":
    run_id = "manual-" + str(uuid.uuid4())

if re.fullmatch("[A-Za-z0-9_-]+", run_id) is None:
    raise ValueError(
        f"Invalid run_id '{run_id}'. "
        "It may contain only letters, digits, - and _."
    )

print("run_id:", run_id)

### 1.5 Is the setup done?

The audit table is created by `08_gold_audit_table`, the Silver tables by the Silver notebooks, and each Gold table by its Gold notebook. If one of the 16 tables does not exist, the notebook stops here with a clear message and writes nothing.

G6 must also have its optional columns, because step 6.5 compares its comment split with G4. A G6 table of the first version gets them when `06_repository_activity_profile` runs once.

In [0]:
if not spark.catalog.tableExists(AUDIT_TABLE):
    raise RuntimeError(
        f"{AUDIT_TABLE} does not exist. Run 08_gold_audit_table first."
    )

silver_tables = [
    PUSH_TABLE,
    PULL_REQUEST_TABLE,
    ISSUE_TABLE,
    ISSUE_COMMENT_TABLE,
    REVIEW_TABLE,
    REVIEW_COMMENT_TABLE,
    WATCH_TABLE,
    FORK_TABLE,
    RELEASE_TABLE,
]

for silver_table in silver_tables:
    if not spark.catalog.tableExists(silver_table):
        raise RuntimeError(
            f"{silver_table} does not exist. Run the Silver notebook of this table first."
        )

# Each Gold table, with the notebook that builds it.
gold_tables = [
    (G1_TABLE, "01_repository_delivery_metrics"),
    (G2_TABLE, "02_repository_issue_flow"),
    (G3_TABLE, "03_repository_contributor_concentration"),
    (G4_TABLE, "04_repository_collaboration_metrics"),
    (G5_TABLE, "05_release_activity_shift"),
    (G6_TABLE, "06_repository_activity_profile"),
]

for gold_table, gold_notebook in gold_tables:
    if not spark.catalog.tableExists(gold_table):
        raise RuntimeError(
            f"{gold_table} does not exist. Run {gold_notebook} first."
        )

if "issue_conversation_comment_events" not in spark.table(G6_TABLE).columns:
    raise RuntimeError(
        f"{G6_TABLE} has no optional columns yet. Run 06_repository_activity_profile once for all dates."
    )

print("Setup found  :", AUDIT_TABLE)
print("Silver tables:", len(silver_tables))
print("Gold tables  :", len(gold_tables))

## 2. The days to compare

The daily tables G1, G2, G3, G4 and G6 are compared day by day:

- **With a blank date,** the whole tables are compared. Every Gold row is read, also a row whose day no longer exists in Silver. Silver gives all rows with an event date.
- **With a `source_date`,** only the days of that source date are compared, on both sides. These are the event dates of the Silver rows that were loaded from that source date, in any of the nine tables. G6 rebuilds exactly these days for that source date, and each other daily table the part of them that has its own kinds.

Three groups of checks always read whole tables, also with a `source_date`:

- **The keys.** A key must be unique in the whole table.
- **G5.** It is one row per release and is always rebuilt completely, so it has no days of its own.
- **Every G5 repository is in G6.** A release looks at the days around it, so this check looks at the whole G6 table.

### 2.1 The Silver rows in the scope of this run

`UNION ALL` places the event dates of the nine tables under each other. A row without an event date cannot be placed on a day. It is counted here, and the count goes into the audit row. The Gold notebooks leave such rows out too.

In [0]:
if source_date_value is None:
    scope_filter = "true"
else:
    scope_filter = f"_source_date = DATE'{source_date}'"

scope_rows = spark.sql(f"""
    SELECT event_date FROM {PUSH_TABLE}           WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {PULL_REQUEST_TABLE}   WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {ISSUE_TABLE}          WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {ISSUE_COMMENT_TABLE}  WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {REVIEW_TABLE}         WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {REVIEW_COMMENT_TABLE} WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {WATCH_TABLE}          WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {FORK_TABLE}           WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {RELEASE_TABLE}        WHERE {scope_filter}
""")
scope_rows.createOrReplaceTempView("scope_rows")

scope_row = spark.sql("""
    SELECT
        COUNT(*)                      AS rows_in_scope,
        count_if(event_date IS NULL)  AS rows_without_event_date
    FROM scope_rows
""").collect()[0]

rows_in_scope = scope_row["rows_in_scope"]
rows_without_event_date = scope_row["rows_without_event_date"]

print("Scope filter              :", scope_filter)
print("Silver rows in the scope  :", rows_in_scope)
print("Rows without an event date:", rows_without_event_date)

### 2.2 The days and the two filters

The days are the event dates of the rows in the scope. When there is no day, there is nothing to compare: the run ends as `NO_INPUT`, and the checks are not run.

Two filters are written as SQL text and used by every later query, so every check compares exactly the same days:

| Filter | Blank date | With a `source_date` |
|---|---|---|
| `silver_day_filter` | `event_date IS NOT NULL` | `event_date IN (the days)` |
| `gold_day_filter` | `true`: every row | `activity_date IN (the days)` |

In [0]:
day_rows = spark.sql("""
    SELECT DISTINCT event_date
    FROM scope_rows
    WHERE event_date IS NOT NULL
    ORDER BY event_date
""").collect()

affected_dates = []
for day_row in day_rows:
    affected_dates.append(day_row["event_date"])

if len(affected_dates) == 0:
    days_sql = "NULL"
else:
    day_texts = []
    for affected_date in affected_dates:
        day_texts.append("DATE'" + str(affected_date) + "'")
    days_sql = ", ".join(day_texts)

if source_date_value is None:
    silver_day_filter = "event_date IS NOT NULL"
    gold_day_filter = "true"
else:
    silver_day_filter = f"event_date IN ({days_sql})"
    gold_day_filter = f"activity_date IN ({days_sql})"

# The checks run only when there is at least one day to compare.
run_checks = len(affected_dates) > 0

# Every check adds one line to this list: its name, what was found, what was expected, and whether it passed.
check_results = []

print("Days to compare:", len(affected_dates))
for affected_date in affected_dates:
    print("  ", affected_date)
print("Silver filter:", silver_day_filter)
print("Gold filter  :", gold_day_filter)
print("Run the checks:", run_checks)

## 3. Read Silver

### 3.1 The Silver rows per table

One query with nine small queries in brackets. Each one counts the rows of one Silver table on the days to compare. The nine numbers go into the audit row, and step 5 compares them with the Gold tables.

In [0]:
silver_row_counts = {
    "push_events": 0,
    "pull_request_events": 0,
    "issue_events": 0,
    "issue_comment_events": 0,
    "pull_request_review_events": 0,
    "pull_request_review_comment_events": 0,
    "watch_events": 0,
    "fork_events": 0,
    "release_events": 0,
}

if run_checks:
    count_row = spark.sql(f"""
        SELECT
            (SELECT COUNT(*) FROM {PUSH_TABLE}           WHERE {silver_day_filter}) AS push_rows,
            (SELECT COUNT(*) FROM {PULL_REQUEST_TABLE}   WHERE {silver_day_filter}) AS pull_request_rows,
            (SELECT COUNT(*) FROM {ISSUE_TABLE}          WHERE {silver_day_filter}) AS issue_rows,
            (SELECT COUNT(*) FROM {ISSUE_COMMENT_TABLE}  WHERE {silver_day_filter}) AS issue_comment_rows,
            (SELECT COUNT(*) FROM {REVIEW_TABLE}         WHERE {silver_day_filter}) AS review_rows,
            (SELECT COUNT(*) FROM {REVIEW_COMMENT_TABLE} WHERE {silver_day_filter}) AS review_comment_rows,
            (SELECT COUNT(*) FROM {WATCH_TABLE}          WHERE {silver_day_filter}) AS watch_rows,
            (SELECT COUNT(*) FROM {FORK_TABLE}           WHERE {silver_day_filter}) AS fork_rows,
            (SELECT COUNT(*) FROM {RELEASE_TABLE}        WHERE {silver_day_filter}) AS release_rows
    """).collect()[0]

    silver_row_counts["push_events"] = count_row["push_rows"]
    silver_row_counts["pull_request_events"] = count_row["pull_request_rows"]
    silver_row_counts["issue_events"] = count_row["issue_rows"]
    silver_row_counts["issue_comment_events"] = count_row["issue_comment_rows"]
    silver_row_counts["pull_request_review_events"] = count_row["review_rows"]
    silver_row_counts["pull_request_review_comment_events"] = count_row["review_comment_rows"]
    silver_row_counts["watch_events"] = count_row["watch_rows"]
    silver_row_counts["fork_events"] = count_row["fork_rows"]
    silver_row_counts["release_events"] = count_row["release_rows"]

print("Silver rows on the days to compare, per table:")
for silver_table_name in silver_row_counts:
    print("  ", silver_table_name, ":", silver_row_counts[silver_table_name])

### 3.2 The Silver events of each Gold table

Each daily Gold table reads its own kinds of events. Here the Silver counts of step 3.1 are added up the same way:

| Table | Kinds |
|---|---|
| G1 | pull request |
| G2 | issue |
| G3 | push, pull request, issue, issue comment, review, inline review comment |
| G4 | issue comment, review, inline review comment |
| G6 | all nine |

In [0]:
silver_events = {
    "G1": silver_row_counts["pull_request_events"],
    "G2": silver_row_counts["issue_events"],
    "G3": silver_row_counts["push_events"]
          + silver_row_counts["pull_request_events"]
          + silver_row_counts["issue_events"]
          + silver_row_counts["issue_comment_events"]
          + silver_row_counts["pull_request_review_events"]
          + silver_row_counts["pull_request_review_comment_events"],
    "G4": silver_row_counts["issue_comment_events"]
          + silver_row_counts["pull_request_review_events"]
          + silver_row_counts["pull_request_review_comment_events"],
    "G6": sum(silver_row_counts.values()),
}

for table_label in silver_events:
    print(table_label, "Silver events:", silver_events[table_label])

### 3.3 The repository-days of each Gold table

A daily Gold table has one row for every repository and day that has at least one of its kinds of events. So its number of rows must be the number of different repository-days in its Silver tables.

The first query places `event_kind`, `repo_id` and `event_date` of the nine tables under each other. The second query counts the different repository-days for each Gold table, each in one small query in brackets. `SELECT DISTINCT repo_id, event_date` keeps every repository-day once.

In [0]:
silver_days = {"G1": 0, "G2": 0, "G3": 0, "G4": 0, "G6": 0}

if run_checks:
    silver_repository_days = spark.sql(f"""
        SELECT 'push' AS event_kind, repo_id, event_date FROM {PUSH_TABLE} WHERE {silver_day_filter}
        UNION ALL
        SELECT 'pull_request', repo_id, event_date FROM {PULL_REQUEST_TABLE} WHERE {silver_day_filter}
        UNION ALL
        SELECT 'issue', repo_id, event_date FROM {ISSUE_TABLE} WHERE {silver_day_filter}
        UNION ALL
        SELECT 'issue_comment', repo_id, event_date FROM {ISSUE_COMMENT_TABLE} WHERE {silver_day_filter}
        UNION ALL
        SELECT 'review', repo_id, event_date FROM {REVIEW_TABLE} WHERE {silver_day_filter}
        UNION ALL
        SELECT 'review_comment', repo_id, event_date FROM {REVIEW_COMMENT_TABLE} WHERE {silver_day_filter}
        UNION ALL
        SELECT 'watch', repo_id, event_date FROM {WATCH_TABLE} WHERE {silver_day_filter}
        UNION ALL
        SELECT 'fork', repo_id, event_date FROM {FORK_TABLE} WHERE {silver_day_filter}
        UNION ALL
        SELECT 'release', repo_id, event_date FROM {RELEASE_TABLE} WHERE {silver_day_filter}
    """)
    silver_repository_days.createOrReplaceTempView("silver_repository_days")

    days_row = spark.sql("""
        SELECT
            (SELECT COUNT(*) FROM (
                SELECT DISTINCT repo_id, event_date FROM silver_repository_days
                WHERE event_kind = 'pull_request'
            ) g1_days) AS g1_days,

            (SELECT COUNT(*) FROM (
                SELECT DISTINCT repo_id, event_date FROM silver_repository_days
                WHERE event_kind = 'issue'
            ) g2_days) AS g2_days,

            (SELECT COUNT(*) FROM (
                SELECT DISTINCT repo_id, event_date FROM silver_repository_days
                WHERE event_kind IN ('push', 'pull_request', 'issue', 'issue_comment', 'review', 'review_comment')
            ) g3_days) AS g3_days,

            (SELECT COUNT(*) FROM (
                SELECT DISTINCT repo_id, event_date FROM silver_repository_days
                WHERE event_kind IN ('issue_comment', 'review', 'review_comment')
            ) g4_days) AS g4_days,

            (SELECT COUNT(*) FROM (
                SELECT DISTINCT repo_id, event_date FROM silver_repository_days
            ) g6_days) AS g6_days
    """).collect()[0]

    silver_days["G1"] = days_row["g1_days"]
    silver_days["G2"] = days_row["g2_days"]
    silver_days["G3"] = days_row["g3_days"]
    silver_days["G4"] = days_row["g4_days"]
    silver_days["G6"] = days_row["g6_days"]

for table_label in silver_days:
    print(table_label, "Silver repository-days:", silver_days[table_label])

### 3.4 All release events, for G5

G5 is always built from all release events, so it is compared with all of them, also with a `source_date`. Here every row counts, also a row without an event date, because G5 counts every release event too: as an anchor or under a reason.

In [0]:
silver_release_events_all_days = 0

if run_checks:
    silver_release_events_all_days = spark.sql(f"""
        SELECT COUNT(*) AS release_events
        FROM {RELEASE_TABLE}
    """).collect()[0]["release_events"]

print("All release events in Silver:", silver_release_events_all_days)

## 4. Keys

Every Gold table has a key: `repo_id` and `activity_date` for the five daily tables, `repo_id` and `release_anchor_key` for G5. The Gold notebooks check their key before they publish. This check reads the published tables, so it also finds a row that came in another way.

Two numbers per table:

- **Keys that appear more than once:** `GROUP BY` the key, and keep the groups with more than one row.
- **Rows with a NULL in the key.**

The six tables are checked with the same query, so a `for` loop runs it once per table. Each table gives two lines in `check_results`, for example `G1_KEY_NOT_UNIQUE` and `G1_KEY_IS_NULL`. The keys are always checked on the whole table.

In [0]:
key_tables = [
    ("G1", G1_TABLE, "repo_id, activity_date", "repo_id IS NULL OR activity_date IS NULL"),
    ("G2", G2_TABLE, "repo_id, activity_date", "repo_id IS NULL OR activity_date IS NULL"),
    ("G3", G3_TABLE, "repo_id, activity_date", "repo_id IS NULL OR activity_date IS NULL"),
    ("G4", G4_TABLE, "repo_id, activity_date", "repo_id IS NULL OR activity_date IS NULL"),
    ("G5", G5_TABLE, "repo_id, release_anchor_key", "repo_id IS NULL OR release_anchor_key IS NULL"),
    ("G6", G6_TABLE, "repo_id, activity_date", "repo_id IS NULL OR activity_date IS NULL"),
]

if run_checks:
    for table_label, table_name, key_columns, null_key_condition in key_tables:
        key_row = spark.sql(f"""
            SELECT
                (SELECT COUNT(*) FROM (
                    SELECT {key_columns}
                    FROM {table_name}
                    GROUP BY {key_columns}
                    HAVING COUNT(*) > 1
                ) keys_seen_twice) AS duplicate_keys,

                (SELECT COUNT(*)
                 FROM {table_name}
                 WHERE {null_key_condition}) AS null_keys
        """).collect()[0]

        duplicate_keys = key_row["duplicate_keys"]
        null_keys = key_row["null_keys"]

        check_results.append((table_label + "_KEY_NOT_UNIQUE", duplicate_keys, 0, duplicate_keys == 0))
        check_results.append((table_label + "_KEY_IS_NULL", null_keys, 0, null_keys == 0))

        print(table_label, "keys that appear more than once:", duplicate_keys, "| rows with a NULL key:", null_keys)

## 5. Gold against Silver

### 5.1 The rows and the events of the daily tables

One query per table counts its rows and adds up its event column on the days to compare:

| Table | Event column |
|---|---|
| G1 | `pr_event_count` |
| G2 | `issue_event_count` |
| G3 | `engineering_activity_events` |
| G4 | `collaboration_events_total` |
| G6 | `all_retained_event_count` |

The rows must be the repository-days of step 3.3, and the events the Silver events of step 3.2. `COALESCE(SUM(...), 0)`: the sum of no rows is NULL, and it has to be 0 here.

In [0]:
gold_rows = {"G1": 0, "G2": 0, "G3": 0, "G4": 0, "G6": 0}
gold_events = {"G1": 0, "G2": 0, "G3": 0, "G4": 0, "G6": 0}

if run_checks:
    g1_row = spark.sql(f"""
        SELECT COUNT(*) AS gold_rows, COALESCE(SUM(pr_event_count), 0) AS gold_events
        FROM {G1_TABLE}
        WHERE {gold_day_filter}
    """).collect()[0]

    g2_row = spark.sql(f"""
        SELECT COUNT(*) AS gold_rows, COALESCE(SUM(issue_event_count), 0) AS gold_events
        FROM {G2_TABLE}
        WHERE {gold_day_filter}
    """).collect()[0]

    g3_row = spark.sql(f"""
        SELECT COUNT(*) AS gold_rows, COALESCE(SUM(engineering_activity_events), 0) AS gold_events
        FROM {G3_TABLE}
        WHERE {gold_day_filter}
    """).collect()[0]

    g4_row = spark.sql(f"""
        SELECT COUNT(*) AS gold_rows, COALESCE(SUM(collaboration_events_total), 0) AS gold_events
        FROM {G4_TABLE}
        WHERE {gold_day_filter}
    """).collect()[0]

    g6_row = spark.sql(f"""
        SELECT COUNT(*) AS gold_rows, COALESCE(SUM(all_retained_event_count), 0) AS gold_events
        FROM {G6_TABLE}
        WHERE {gold_day_filter}
    """).collect()[0]

    gold_rows["G1"] = g1_row["gold_rows"]
    gold_rows["G2"] = g2_row["gold_rows"]
    gold_rows["G3"] = g3_row["gold_rows"]
    gold_rows["G4"] = g4_row["gold_rows"]
    gold_rows["G6"] = g6_row["gold_rows"]

    gold_events["G1"] = g1_row["gold_events"]
    gold_events["G2"] = g2_row["gold_events"]
    gold_events["G3"] = g3_row["gold_events"]
    gold_events["G4"] = g4_row["gold_events"]
    gold_events["G6"] = g6_row["gold_events"]

    for table_label in ["G1", "G2", "G3", "G4", "G6"]:
        check_results.append((
            table_label + "_ROWS_DIFFER_FROM_SILVER",
            gold_rows[table_label],
            silver_days[table_label],
            gold_rows[table_label] == silver_days[table_label],
        ))
        check_results.append((
            table_label + "_EVENTS_DIFFER_FROM_SILVER",
            gold_events[table_label],
            silver_events[table_label],
            gold_events[table_label] == silver_events[table_label],
        ))

for table_label in ["G1", "G2", "G3", "G4", "G6"]:
    print(
        table_label,
        "rows:", gold_rows[table_label], "(Silver", str(silver_days[table_label]) + ")",
        "| events:", gold_events[table_label], "(Silver", str(silver_events[table_label]) + ")",
    )

### 5.2 The nine kinds of G6

G6 has one column per kind of event. The sum of each column must be the number of rows of its Silver table. The nine sums are put into a dictionary with the same names as `silver_row_counts` of step 3.1, and the check counts the kinds whose numbers differ.

In [0]:
g6_kind_sums = {}
kinds_that_differ = 0

if run_checks:
    kind_row = spark.sql(f"""
        SELECT
            COALESCE(SUM(push_events), 0)                   AS push_events,
            COALESCE(SUM(pull_request_events), 0)           AS pull_request_events,
            COALESCE(SUM(issue_events), 0)                  AS issue_events,
            COALESCE(SUM(issue_comment_events), 0)          AS issue_comment_events,
            COALESCE(SUM(review_events), 0)                 AS review_events,
            COALESCE(SUM(inline_review_comment_events), 0)  AS inline_review_comment_events,
            COALESCE(SUM(watch_events), 0)                  AS watch_events,
            COALESCE(SUM(fork_events), 0)                   AS fork_events,
            COALESCE(SUM(release_events), 0)                AS release_events
        FROM {G6_TABLE}
        WHERE {gold_day_filter}
    """).collect()[0]

    # The names of the Silver tables, so that the two dictionaries can be compared name by name.
    g6_kind_sums = {
        "push_events": kind_row["push_events"],
        "pull_request_events": kind_row["pull_request_events"],
        "issue_events": kind_row["issue_events"],
        "issue_comment_events": kind_row["issue_comment_events"],
        "pull_request_review_events": kind_row["review_events"],
        "pull_request_review_comment_events": kind_row["inline_review_comment_events"],
        "watch_events": kind_row["watch_events"],
        "fork_events": kind_row["fork_events"],
        "release_events": kind_row["release_events"],
    }

    for silver_table_name in silver_row_counts:
        print(
            "  ", silver_table_name,
            "| G6:", g6_kind_sums[silver_table_name],
            "| Silver:", silver_row_counts[silver_table_name],
        )
        if g6_kind_sums[silver_table_name] != silver_row_counts[silver_table_name]:
            kinds_that_differ = kinds_that_differ + 1

    check_results.append(("G6_KIND_DIFFERS_FROM_SILVER", kinds_that_differ, 0, kinds_that_differ == 0))

print("Kinds whose G6 sum differs from Silver:", kinds_that_differ)

### 5.3 G5: its rows and the release events it left out

G5 checks every release event of Silver against seven rules. An event that passes all seven is an anchor and gets a row. Every other event is counted under one reason, and the seven counts are in the audit row of the G5 run. So:

**rows of G5 + release events left out by its last successful run = all release events of Silver**

- The last successful run is the newest audit row of G5 with the status `SUCCEEDED`, for this G5 table.
- When there is no such run, there is nothing to add up, and the check fails.
- When Silver got new release events after that run, the sum is too small and the check fails too: G5 has to run again.

In [0]:
g5_row_count = 0
g5_left_out = None
g5_total = None

if run_checks:
    g5_row_count = spark.sql(f"""
        SELECT COUNT(*) AS g5_rows
        FROM {G5_TABLE}
    """).collect()[0]["g5_rows"]

    g5_runs = spark.sql(f"""
        SELECT quality_metrics
        FROM {AUDIT_TABLE}
        WHERE pipeline_name = '{G5_PIPELINE_NAME}'
          AND gold_table = '{G5_TABLE}'
          AND status = 'SUCCEEDED'
        ORDER BY run_finished_at DESC
        LIMIT 1
    """).collect()

    if len(g5_runs) > 0:
        g5_metrics = g5_runs[0]["quality_metrics"]
        g5_left_out = (
            g5_metrics["NOT_PUBLISHED"]
            + g5_metrics["DRAFT_OR_UNKNOWN"]
            + g5_metrics["MISSING_PUBLISHED_AT"]
            + g5_metrics["PRERELEASE_UNKNOWN"]
            + g5_metrics["MISSING_RELEASE_ID"]
            + g5_metrics["REPEATED_RELEASE_EVENT"]
            + g5_metrics["WINDOW_NOT_COVERED"]
        )
        g5_total = g5_row_count + g5_left_out

    check_results.append((
        "G5_DOES_NOT_ADD_UP_TO_SILVER",
        g5_total,
        silver_release_events_all_days,
        g5_total == silver_release_events_all_days,
    ))

print("G5 rows                        :", g5_row_count)
print("Left out by the last G5 run    :", g5_left_out)
print("Together                       :", g5_total)
print("All release events in Silver   :", silver_release_events_all_days)

## 6. Gold against Gold

G6 counts all nine kinds, so on every repository and day it must agree with the other daily tables:

| Check | G6 | must equal |
|---|---|---|
| 6.1 | `pull_request_events` | G1 `pr_event_count` |
| 6.2 | `issue_events` | G2 `issue_event_count` |
| 6.3 | push + pull request + issue + issue comment + review + inline review comment events | G3 `engineering_activity_events` |
| 6.4 | issue comment + review + inline review comment events | G4 `collaboration_events_total` |
| 6.5 | the three comment columns: on issues, in pull request conversations, unknown | the same three columns of G4 |

**How a check finds a difference.** `FULL OUTER JOIN` puts the two tables side by side on `repo_id` and `activity_date`. It keeps every row of both tables: also a row that has no partner on the other side. Then the missing side is NULL, and `COALESCE(..., 0)` turns it into 0. So a G6 row with 2 pull request events and no G1 row is a difference, and so is a G1 row without a G6 row. A G6 row with 0 pull request events and no G1 row is not.

Each check counts the repository-days that differ. It must be 0.

### 6.1 G6 against G1: pull request events

In [0]:
if run_checks:
    g1_difference = spark.sql(f"""
        SELECT COUNT(*) AS rows_that_differ
        FROM (SELECT * FROM {G1_TABLE} WHERE {gold_day_filter}) g1
        FULL OUTER JOIN (SELECT * FROM {G6_TABLE} WHERE {gold_day_filter}) g6
            ON  g1.repo_id = g6.repo_id
            AND g1.activity_date = g6.activity_date
        WHERE COALESCE(g1.pr_event_count, 0) <> COALESCE(g6.pull_request_events, 0)
    """).collect()[0]["rows_that_differ"]

    check_results.append(("G6_PULL_REQUESTS_DIFFER_FROM_G1", g1_difference, 0, g1_difference == 0))
    print("Repository-days where G6 and G1 differ:", g1_difference)

### 6.2 G6 against G2: issue events

In [0]:
if run_checks:
    g2_difference = spark.sql(f"""
        SELECT COUNT(*) AS rows_that_differ
        FROM (SELECT * FROM {G2_TABLE} WHERE {gold_day_filter}) g2
        FULL OUTER JOIN (SELECT * FROM {G6_TABLE} WHERE {gold_day_filter}) g6
            ON  g2.repo_id = g6.repo_id
            AND g2.activity_date = g6.activity_date
        WHERE COALESCE(g2.issue_event_count, 0) <> COALESCE(g6.issue_events, 0)
    """).collect()[0]["rows_that_differ"]

    check_results.append(("G6_ISSUES_DIFFER_FROM_G2", g2_difference, 0, g2_difference == 0))
    print("Repository-days where G6 and G2 differ:", g2_difference)

### 6.3 G6 against G3: the six engineering kinds

In [0]:
if run_checks:
    g3_difference = spark.sql(f"""
        SELECT COUNT(*) AS rows_that_differ
        FROM (SELECT * FROM {G3_TABLE} WHERE {gold_day_filter}) g3
        FULL OUTER JOIN (SELECT * FROM {G6_TABLE} WHERE {gold_day_filter}) g6
            ON  g3.repo_id = g6.repo_id
            AND g3.activity_date = g6.activity_date
        WHERE COALESCE(g3.engineering_activity_events, 0)
           <> COALESCE(g6.push_events
                       + g6.pull_request_events
                       + g6.issue_events
                       + g6.issue_comment_events
                       + g6.review_events
                       + g6.inline_review_comment_events, 0)
    """).collect()[0]["rows_that_differ"]

    check_results.append(("G6_ENGINEERING_DIFFERS_FROM_G3", g3_difference, 0, g3_difference == 0))
    print("Repository-days where G6 and G3 differ:", g3_difference)

### 6.4 G6 against G4: the three collaboration kinds

In [0]:
if run_checks:
    g4_difference = spark.sql(f"""
        SELECT COUNT(*) AS rows_that_differ
        FROM (SELECT * FROM {G4_TABLE} WHERE {gold_day_filter}) g4
        FULL OUTER JOIN (SELECT * FROM {G6_TABLE} WHERE {gold_day_filter}) g6
            ON  g4.repo_id = g6.repo_id
            AND g4.activity_date = g6.activity_date
        WHERE COALESCE(g4.collaboration_events_total, 0)
           <> COALESCE(g6.issue_comment_events
                       + g6.review_events
                       + g6.inline_review_comment_events, 0)
    """).collect()[0]["rows_that_differ"]

    check_results.append(("G6_COLLABORATION_DIFFERS_FROM_G4", g4_difference, 0, g4_difference == 0))
    print("Repository-days where G6 and G4 differ:", g4_difference)

### 6.5 G6 against G4: the comment split

An issue comment is written on an issue, in the conversation of a pull request, or on an unknown target. G4 and G6 both count the three groups. A row differs when one of the three numbers differs.

In [0]:
if run_checks:
    split_difference = spark.sql(f"""
        SELECT COUNT(*) AS rows_that_differ
        FROM (SELECT * FROM {G4_TABLE} WHERE {gold_day_filter}) g4
        FULL OUTER JOIN (SELECT * FROM {G6_TABLE} WHERE {gold_day_filter}) g6
            ON  g4.repo_id = g6.repo_id
            AND g4.activity_date = g6.activity_date
        WHERE COALESCE(g4.issue_conversation_comment_events, 0)        <> COALESCE(g6.issue_conversation_comment_events, 0)
           OR COALESCE(g4.pr_conversation_comment_events, 0)           <> COALESCE(g6.pr_conversation_comment_events, 0)
           OR COALESCE(g4.unclassified_conversation_comment_events, 0) <> COALESCE(g6.unclassified_conversation_comment_events, 0)
    """).collect()[0]["rows_that_differ"]

    check_results.append(("G6_COMMENT_SPLIT_DIFFERS_FROM_G4", split_difference, 0, split_difference == 0))
    print("Repository-days where the comment split of G6 and G4 differs:", split_difference)

### 6.6 The people: G6 ≥ G3 ≥ G4

On one repository and day, G6 counts the different people of nine kinds of events, G3 of six of them, and G4 of three of these six. More kinds can only bring more people, never fewer. So on every repository-day:

**people in G6 ≥ people in G3 ≥ people in G4**

| Table | Column |
|---|---|
| G6 | `unique_known_actors` |
| G3 | `unique_active_actors` |
| G4 | `unique_collaborating_actors` |

Every G3 row is compared with its G6 row, and every G4 row with its G3 row. `LEFT JOIN` keeps every row of the left table. When the row on the right is missing, its people count as 0. A right row without a left partner cannot break the order, so it is not needed here.

In [0]:
if run_checks:
    g6_below_g3 = spark.sql(f"""
        SELECT COUNT(*) AS rows_out_of_order
        FROM (SELECT * FROM {G3_TABLE} WHERE {gold_day_filter}) g3
        LEFT JOIN (SELECT * FROM {G6_TABLE} WHERE {gold_day_filter}) g6
            ON  g3.repo_id = g6.repo_id
            AND g3.activity_date = g6.activity_date
        WHERE COALESCE(g6.unique_known_actors, 0) < g3.unique_active_actors
    """).collect()[0]["rows_out_of_order"]

    g3_below_g4 = spark.sql(f"""
        SELECT COUNT(*) AS rows_out_of_order
        FROM (SELECT * FROM {G4_TABLE} WHERE {gold_day_filter}) g4
        LEFT JOIN (SELECT * FROM {G3_TABLE} WHERE {gold_day_filter}) g3
            ON  g4.repo_id = g3.repo_id
            AND g4.activity_date = g3.activity_date
        WHERE COALESCE(g3.unique_active_actors, 0) < g4.unique_collaborating_actors
    """).collect()[0]["rows_out_of_order"]

    check_results.append(("ACTORS_G6_BELOW_G3", g6_below_g3, 0, g6_below_g3 == 0))
    check_results.append(("ACTORS_G3_BELOW_G4", g3_below_g4, 0, g3_below_g4 == 0))
    print("Repository-days with fewer people in G6 than in G3:", g6_below_g3)
    print("Repository-days with fewer people in G3 than in G4:", g3_below_g4)

### 6.7 Every row of G1 to G4 is in G6

A repository-day with an event in G1, G2, G3 or G4 has that event in G6 too, so it must have a G6 row.

`UNION ALL` places the keys of the four tables under each other. `LEFT JOIN` looks for the G6 row of each key, and `WHERE g6.repo_id IS NULL` keeps the keys without one. A key that is missing from G6 is counted once for every table it comes from.

In [0]:
if run_checks:
    rows_not_in_g6 = spark.sql(f"""
        SELECT COUNT(*) AS rows_not_in_g6
        FROM (
            SELECT repo_id, activity_date FROM {G1_TABLE} WHERE {gold_day_filter}
            UNION ALL
            SELECT repo_id, activity_date FROM {G2_TABLE} WHERE {gold_day_filter}
            UNION ALL
            SELECT repo_id, activity_date FROM {G3_TABLE} WHERE {gold_day_filter}
            UNION ALL
            SELECT repo_id, activity_date FROM {G4_TABLE} WHERE {gold_day_filter}
        ) daily_rows
        LEFT JOIN (SELECT repo_id, activity_date FROM {G6_TABLE} WHERE {gold_day_filter}) g6
            ON  daily_rows.repo_id = g6.repo_id
            AND daily_rows.activity_date = g6.activity_date
        WHERE g6.repo_id IS NULL
    """).collect()[0]["rows_not_in_g6"]

    check_results.append(("G1_TO_G4_ROW_NOT_IN_G6", rows_not_in_g6, 0, rows_not_in_g6 == 0))
    print("Rows of G1 to G4 without a G6 row:", rows_not_in_g6)

### 6.8 Every G5 repository is in G6

A release event is one of the nine kinds of G6. So a repository with an anchor in G5 has at least one row in G6. This check reads the whole G6 table, also with a `source_date`, because G5 has no days of its own.

In [0]:
if run_checks:
    repositories_not_in_g6 = spark.sql(f"""
        SELECT COUNT(DISTINCT g5.repo_id) AS repositories_not_in_g6
        FROM {G5_TABLE} g5
        LEFT JOIN (SELECT DISTINCT repo_id FROM {G6_TABLE}) g6
            ON g5.repo_id = g6.repo_id
        WHERE g6.repo_id IS NULL
    """).collect()[0]["repositories_not_in_g6"]

    check_results.append(("G5_REPOSITORY_NOT_IN_G6", repositories_not_in_g6, 0, repositories_not_in_g6 == 0))
    print("G5 repositories without a G6 row:", repositories_not_in_g6)

## 7. The result

### 7.1 All checks in one table

Every check has added one line to `check_results`. Here they become one small table, in the order in which they ran. `found` is what the Gold tables gave, and `expected` what it must be: 0, or the number from Silver.

In [0]:
failed_checks = []

for check_name, found, expected, passed in check_results:
    if not passed:
        failed_checks.append(check_name)

if run_checks:
    check_table = spark.createDataFrame(
        check_results,
        "check_name STRING, found BIGINT, expected BIGINT, passed BOOLEAN",
    )
    display(check_table)

print("Checks run   :", len(check_results))
print("Checks failed:", len(failed_checks))
for check_name in failed_checks:
    print("   FAILED:", check_name)

### 7.2 The status of the run

| Situation | Status |
|---|---|
| There is no day to compare, because Silver has no rows in the scope of this run | `NO_INPUT` |
| A check failed | `FAILED` |
| Every check passed | `SUCCEEDED` |

In [0]:
error_message = None

if not run_checks:
    run_status = "NO_INPUT"
elif len(failed_checks) > 0:
    run_status = "FAILED"
    error_message = "Checks failed: " + "; ".join(failed_checks)
else:
    run_status = "SUCCEEDED"

print("Status:", run_status)

## 8. Audit

One row is added to the Gold audit table. It has the same 18 columns as the rows of the Gold notebooks. Some of them mean something a little different here, because this notebook writes no Gold table:

| Column | Value |
|---|---|
| `gold_table` | NULL: no table was written |
| `affected_dates` | The days that were compared |
| `gold_rows_published` | 0 |
| `gold_table_version` | NULL |
| `source_rows_read` | The Silver rows of step 3.1, one number per table |
| `rows_excluded` | `missing_event_date`: Silver rows in the scope without an event date |
| `quality_metrics` | The number of checks and of failed checks, and the rows and events of each Gold table that were compared |

If a check failed, the audit row is written first. Then the notebook stops with an error, so that the Job task fails too.

In [0]:
if len(failed_checks) == 0:
    failed_checks_text = None
else:
    failed_checks_text = "; ".join(failed_checks)

quality_metrics = {
    "checks_run": len(check_results),
    "checks_failed": len(failed_checks),
    "g1_rows": gold_rows["G1"],
    "g1_events": gold_events["G1"],
    "g2_rows": gold_rows["G2"],
    "g2_events": gold_events["G2"],
    "g3_rows": gold_rows["G3"],
    "g3_events": gold_events["G3"],
    "g4_rows": gold_rows["G4"],
    "g4_events": gold_events["G4"],
    "g5_rows": g5_row_count,
    "g5_left_out_release_events": g5_left_out,
    "g6_rows": gold_rows["G6"],
    "g6_events": gold_events["G6"],
    "silver_release_events_all_days": silver_release_events_all_days,
}

run_finished_at = datetime.now(timezone.utc)

audit_row = (
    str(uuid.uuid4()),                                  # audit_id
    run_id,                                             # run_id
    PIPELINE_NAME,                                      # pipeline_name
    None,                                               # gold_table: no table was written
    processing_scope,                                   # processing_scope
    source_date_value,                                  # source_date
    affected_dates,                                     # affected_dates: the days compared
    run_started_at,                                     # run_started_at
    run_finished_at,                                    # run_finished_at
    run_status,                                         # status
    error_message,                                      # error_message
    silver_row_counts,                                  # source_rows_read
    {"missing_event_date": rows_without_event_date},    # rows_excluded
    0,                                                  # gold_rows_published
    None,                                               # gold_table_version
    len(failed_checks) == 0,                            # checks_passed
    failed_checks_text,                                 # failed_checks
    quality_metrics,                                    # quality_metrics
)

audit_schema = spark.table(AUDIT_TABLE).schema
audit_df = spark.createDataFrame([audit_row], schema=audit_schema)
audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)

print("Audit row written:", AUDIT_TABLE)
print("run_id:", run_id)
print("status:", run_status)

if run_status == "NO_INPUT":
    print("WARNING: Silver has no rows in the scope of this run. Nothing was compared.")

if rows_without_event_date > 0:
    print("WARNING: Silver rows without an event date:", rows_without_event_date)

if run_status == "FAILED":
    raise RuntimeError("gold_final_audit FAILED: " + error_message)

## 9. Results

These cells only read.

### 9.1 The audit row of this run

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {AUDIT_TABLE}
    WHERE run_id = '{run_id}'
      AND pipeline_name = '{PIPELINE_NAME}'
"""))

### 9.2 Each Gold table next to Silver

The numbers of steps 3 and 5 side by side. After a run for all dates they can be compared with the expected values of the full week in section 15 of the contract.

In [0]:
summary_rows = []
for table_label in ["G1", "G2", "G3", "G4", "G6"]:
    summary_rows.append((
        table_label,
        gold_rows[table_label],
        silver_days[table_label],
        gold_events[table_label],
        silver_events[table_label],
    ))

if run_checks:
    display(spark.createDataFrame(
        summary_rows,
        "gold_table STRING, gold_rows BIGINT, silver_repository_days BIGINT, gold_events BIGINT, silver_events BIGINT",
    ))

print("G5 rows:", g5_row_count, "| left out by its last run:", g5_left_out, "| all release events in Silver:", silver_release_events_all_days)

## What this notebook wrote

| Table | What |
|---|---|
| `audit.gold_pipeline_runs` | One row for this run |

No Gold table and no Silver table was changed.

## How to read the result

- `SUCCEEDED` means that the six tables agree with Silver and with each other on the days that were compared. It does not prove that a single value inside a row, such as a median or a share, is right. That is what the test notebook of each Gold table proves.
- When a check fails, its line in step 7.1 shows the number that was found and the number that was expected. The usual reason is a Gold table that was not rebuilt after Silver changed: run its Gold notebook again for the same source date, then this notebook.
- A run with a `source_date` compares only the days of that source date. A run with a blank date compares the whole tables.